# GaussianPro + Scaffold-GS — train toàn bộ scene (30k)

Notebook train tuần tự danh sách scene khai báo rõ ràng trong `acomingz/maindataset`, dùng **100% ảnh COLMAP**, render toàn bộ `test_poses.csv`, rồi tạo một ZIP submission ngay khi từng scene hoàn tất. Không có dashboard/monitor định kỳ và không dùng WandB.

Bật GPU trong **Settings → Accelerator** trước khi chạy. Notebook cần dataset chứa source code hiện tại (file `kusanagi-source.zip` hoặc cây source có `train.py`) được attach cùng kernel.

In [ ]:
from pathlib import Path
import csv, gc, json, os, shutil, subprocess, sys, time, traceback, zipfile

# ========================= CẤU HÌNH =========================
DATA_ROOT = Path('/kaggle/input/datasets/acomingz/maindataset')
WORK_ROOT = Path('/kaggle/working/gaussianpro_full_30k')
MODEL_ROOT = WORK_ROOT / 'models'
ZIP_ROOT = Path('/kaggle/working/scene_zips')
REPO_DIR = Path('/kaggle/working/kusanagi')

ITERATIONS = 30_000
RESOLUTION = 1
DATA_DEVICE = 'cpu'          # giảm VRAM; ảnh được chuyển lên GPU khi cần
GPU = '0'
SKIP_FINISHED = True         # chạy lại notebook sẽ bỏ qua ZIP đã hoàn chỉnh
DELETE_MODEL_AFTER_ZIP = True # tránh đầy /kaggle/working khi có nhiều scene
CORRECT_RADIAL_DISTORTION = True

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
print('Data:', DATA_ROOT)
print('ZIP output:', ZIP_ROOT)


In [ ]:
# Tìm và chuẩn bị source code hiện tại.
def valid_repo(path: Path) -> bool:
    return (path / 'train.py').is_file() and (path / 'render.py').is_file()

if not valid_repo(REPO_DIR):
    source_roots = [p.parent for p in Path('/kaggle/input').rglob('train.py') if valid_repo(p.parent)]
    if source_roots:
        source = sorted(source_roots, key=lambda p: len(p.parts))[0]
        shutil.copytree(source, REPO_DIR, dirs_exist_ok=True)
    else:
        archives = list(Path('/kaggle/input').rglob('kusanagi-source.zip'))
        if len(archives) != 1:
            raise RuntimeError(
                'Không tìm thấy source code. Hãy attach dataset source chứa train.py '
                f'hoặc đúng một kusanagi-source.zip. Archives tìm thấy: {archives}'
            )
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall(REPO_DIR)
        # Hỗ trợ ZIP có thêm một thư mục gốc.
        if not valid_repo(REPO_DIR):
            nested = [p.parent for p in REPO_DIR.rglob('train.py') if valid_repo(p.parent)]
            if len(nested) != 1:
                raise RuntimeError(f'Source ZIP không có repo hợp lệ duy nhất: {nested}')
            source = nested[0]
            target = REPO_DIR.with_name(REPO_DIR.name + '_flat')
            shutil.copytree(source, target, dirs_exist_ok=True)
            REPO_DIR = target

print('Repo:', REPO_DIR)
subprocess.run(['nvidia-smi'], check=True)

# Build hai CUDA extension trên worker sạch.
try:
    import diff_gaussian_rasterization, simple_knn
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        str(REPO_DIR / 'submodules/diff-gaussian-rasterization'),
        str(REPO_DIR / 'submodules/simple-knn'),
        '--no-build-isolation'
    ], check=True)


In [ ]:
# Danh sách và thứ tự scene cần train. Có thể comment một dòng để bỏ scene đó.
SCENES = [
    'chair',
    'bonsai',
    'HCM0421',
    'HCM0539',
    'HCM0540',
    'HCM0644',
    'HCM0674',
]
print('Scene sẽ train:', SCENES)

def is_train_root(path: Path) -> bool:
    has_cameras = (path / 'sparse').is_dir() or (path / 'transforms_train.json').is_file()
    return (path / 'images').is_dir() and has_cameras

def find_pose_csv(source: Path) -> Path:
    candidates = (source / 'test/test_poses.csv', source.parent / 'test/test_poses.csv')
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(f'Không tìm thấy test/test_poses.csv cho {source}')

if not DATA_ROOT.is_dir():
    # Kaggle đôi khi mount dataset bằng slug ngắn thay vì đường dẫn API.
    fallback = Path('/kaggle/input/maindataset')
    if fallback.is_dir():
        DATA_ROOT = fallback
    else:
        raise FileNotFoundError(f'Không tồn tại {DATA_ROOT} hoặc {fallback}')

scenes = {}
invalid_scenes = []
for scene_name in SCENES:
    scene_dir = DATA_ROOT / scene_name
    if not scene_dir.is_dir():
        invalid_scenes.append(f'{scene_name}: không tồn tại {scene_dir}')
        continue
    roots = [root for root in (scene_dir / 'train', scene_dir) if is_train_root(root)]
    if not roots:
        invalid_scenes.append(f'{scene_name}: không có images + sparse')
        continue
    source = roots[0]
    try:
        pose_csv = find_pose_csv(source)
    except FileNotFoundError as error:
        invalid_scenes.append(f'{scene_name}: {error}')
        continue
    image_count = sum(p.is_file() for p in (source / 'images').iterdir())
    with pose_csv.open(newline='', encoding='utf-8-sig') as handle:
        test_count = sum(1 for _ in csv.DictReader(handle))
    scenes[scene_name] = source
    print(f'{scene_name:16s} train={image_count:4d} test={test_count:4d}  {source}')

if invalid_scenes:
    raise RuntimeError('Danh sách scene không hợp lệ:\n- ' + '\n- '.join(invalid_scenes))
if len(scenes) != len(SCENES):
    raise RuntimeError(f'Chỉ resolve được {len(scenes)}/{len(SCENES)} scene')
print(f'\nĐã xác minh đủ {len(scenes)} scene; sẽ train đúng thứ tự trên.')


In [ ]:
# Cấu hình production: lịch 30k, dùng toàn bộ pipeline GaussianPro đã tích hợp.
GAUSSIANPRO_ARGS = [
    '--use_gaussianpro',
    '--gaussianpro_start_iter', '3000',
    '--gaussianpro_add_until_iter', '15000',
    '--gaussianpro_refine_until_iter', '24000',
    '--gaussianpro_interval', '50',
    '--gaussianpro_references_per_step', '2',
    '--gaussianpro_neighbors', '4',
    '--gaussianpro_downsample', '4',
    '--gaussianpro_patch_radius', '2',
    '--gaussianpro_patchmatch_iterations', '3',
    '--gaussianpro_min_consistent_views', '3',
    '--gaussianpro_relaxed_min_views', '2',
    '--gaussianpro_max_photo_error', '0.25',
    '--gaussianpro_reprojection_threshold', '2.0',
    '--gaussianpro_depth_consistency_threshold', '0.03',
    '--gaussianpro_normal_consistency_threshold', '0.5',
    '--gaussianpro_depth_discrepancy_start', '1.0',
    '--gaussianpro_depth_discrepancy_end', '0.8',
    '--gaussianpro_max_anchors_per_step', '128',
    '--gaussianpro_max_anchor_multiplier', '1.25',
    '--lambda_gaussianpro_flatness', '0.001',
    '--lambda_gaussianpro_anchor_normal', '0.001',
    '--lambda_gaussianpro_normal_l1', '0.001',
    '--lambda_gaussianpro_normal_cos', '0.001',
    '--lambda_gaussianpro_feature_l1', '0.0002',
    '--lambda_gaussianpro_feature_cos', '0.0002',
]
RADIAL_ARGS = ['--correct_radial_distortion'] if CORRECT_RADIAL_DISTORTION else []

def final_ply(model_dir: Path) -> Path:
    return model_dir / 'point_cloud' / f'iteration_{ITERATIONS}' / 'point_cloud.ply'

def run_logged(command, cwd: Path, log_path: Path):
    print(' '.join(map(str, command)))
    with log_path.open('w', encoding='utf-8') as log:
        result = subprocess.run(command, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, text=True)
    if result.returncode != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace')[-20000:]
        print(tail)
        raise RuntimeError(f'Lệnh thất bại ({result.returncode}); xem {log_path}')

def train_scene(scene_name: str, source: Path, model_dir: Path):
    if model_dir.exists():
        shutil.rmtree(model_dir)
    model_dir.mkdir(parents=True)
    command = [
        sys.executable, 'train.py',
        '-s', str(source), '-m', str(model_dir),
        '--images', 'images', '-r', str(RESOLUTION),
        '--data_device', DATA_DEVICE, '--appearance_dim', '0', '--gpu', GPU,
        '--iterations', str(ITERATIONS),
        '--validation_ratio', '0.0', '--validation_sample_count', '0',
        '--test_iterations', str(ITERATIONS),
        '--save_iterations', str(ITERATIONS),
        '--lambda_dssim', '0.2', '--lambda_edge_init', '0.0', '--lambda_edge_final', '0.0',
        '--quiet',
    ] + GAUSSIANPRO_ARGS + RADIAL_ARGS
    run_logged(command, REPO_DIR, model_dir / 'train_stdout.log')
    if not final_ply(model_dir).is_file():
        raise RuntimeError(f'{scene_name}: thiếu checkpoint {final_ply(model_dir)}')

def render_scene(source: Path, model_dir: Path) -> Path:
    command = [
        sys.executable, 'render.py',
        '-s', str(source), '-m', str(model_dir), '--iteration', str(ITERATIONS),
        '-r', str(RESOLUTION), '--data_device', DATA_DEVICE,
        '--eval', '--skip_train', '--validation_ratio', '0.0',
    ] + RADIAL_ARGS
    run_logged(command, REPO_DIR, model_dir / 'render_stdout.log')
    render_dir = model_dir / 'test' / f'ours_{ITERATIONS}' / 'renders'
    if not render_dir.is_dir():
        raise RuntimeError(f'Không có render output: {render_dir}')
    return render_dir


In [ ]:
# Kiểm tra render và ZIP nguyên tử: file .tmp chỉ được đổi tên khi archive hợp lệ.
def zip_scene(scene_name: str, source: Path, render_dir: Path) -> Path:
    pose_csv = find_pose_csv(source)
    with pose_csv.open(newline='', encoding='utf-8-sig') as handle:
        rows = list(csv.DictReader(handle))
    expected = [row['image_name'].strip() for row in rows]
    if not expected or len(expected) != len(set(expected)):
        raise RuntimeError(f'{scene_name}: image_name rỗng hoặc trùng trong test_poses.csv')

    actual = {p.name: p for p in render_dir.iterdir() if p.is_file()}
    missing = sorted(set(expected) - set(actual))
    extra = sorted(set(actual) - set(expected))
    if missing or extra:
        raise RuntimeError(f'{scene_name}: render missing={missing}, extra={extra}')

    final_zip = ZIP_ROOT / f'submission_{scene_name}_{ITERATIONS}.zip'
    temp_zip = ZIP_ROOT / f'.{final_zip.name}.tmp'
    if temp_zip.exists():
        temp_zip.unlink()
    entries = [(Path('submission') / scene_name / name).as_posix() for name in expected]
    with zipfile.ZipFile(temp_zip, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for name, archive_name in zip(expected, entries):
            archive.write(actual[name], arcname=archive_name)
    with zipfile.ZipFile(temp_zip) as archive:
        bad_file = archive.testzip()
        archived_names = archive.namelist()
    if bad_file is not None or archived_names != entries:
        temp_zip.unlink(missing_ok=True)
        raise RuntimeError(f'{scene_name}: ZIP audit lỗi, bad_file={bad_file}')
    temp_zip.replace(final_zip)
    print(f'ZIP READY: {final_zip} ({final_zip.stat().st_size / 1024**2:.2f} MiB)')
    return final_zip

def zip_is_valid(scene_name: str, path: Path) -> bool:
    try:
        with zipfile.ZipFile(path) as archive:
            names = archive.namelist()
            return archive.testzip() is None and bool(names) and all(
                name.startswith(f'submission/{scene_name}/') for name in names
            )
    except (OSError, zipfile.BadZipFile):
        return False


In [ ]:
# Pipeline tuần tự: train -> render -> audit -> ZIP -> dọn model -> scene tiếp theo.
results, failures = [], []
for index, (scene_name, source) in enumerate(scenes.items(), start=1):
    zip_path = ZIP_ROOT / f'submission_{scene_name}_{ITERATIONS}.zip'
    model_dir = MODEL_ROOT / scene_name
    print(f'\n========== [{index}/{len(scenes)}] {scene_name} ==========')
    if SKIP_FINISHED and zip_path.is_file() and zip_is_valid(scene_name, zip_path):
        print('[SKIP] ZIP hợp lệ đã tồn tại:', zip_path)
        results.append({'scene': scene_name, 'status': 'skipped', 'zip': str(zip_path)})
        continue

    started = time.time()
    try:
        train_scene(scene_name, source, model_dir)
        render_dir = render_scene(source, model_dir)
        zip_path = zip_scene(scene_name, source, render_dir)
        elapsed = round((time.time() - started) / 60, 1)
        results.append({
            'scene': scene_name, 'status': 'completed', 'minutes': elapsed,
            'zip_MiB': round(zip_path.stat().st_size / 1024**2, 2), 'zip': str(zip_path)
        })
        if DELETE_MODEL_AFTER_ZIP and zip_is_valid(scene_name, zip_path):
            shutil.rmtree(model_dir)
            print('Đã dọn model trung gian sau khi xác minh ZIP.')
    except Exception as error:
        traceback.print_exc()
        failures.append({'scene': scene_name, 'error': repr(error)})
        results.append({'scene': scene_name, 'status': 'failed', 'error': repr(error)})
        print(f'[FAILED] {scene_name}; tiếp tục scene kế tiếp.')
    finally:
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
        (WORK_ROOT / 'run_status.json').write_text(json.dumps(results, indent=2), encoding='utf-8')

print('\n========== TỔNG KẾT ==========')
print(json.dumps(results, indent=2, ensure_ascii=False))
if failures:
    raise RuntimeError(f'{len(failures)} scene thất bại: {failures}')
print('ALL SCENES COMPLETED — ZIP directory:', ZIP_ROOT)


In [ ]:
# Audit cuối: mọi scene phải có đúng một ZIP đọc được.
audit = []
for scene_name in scenes:
    path = ZIP_ROOT / f'submission_{scene_name}_{ITERATIONS}.zip'
    if not zip_is_valid(scene_name, path):
        raise RuntimeError(f'ZIP thiếu hoặc hỏng: {path}')
    with zipfile.ZipFile(path) as archive:
        image_count = len(archive.namelist())
    audit.append({
        'scene': scene_name, 'images': image_count,
        'MiB': round(path.stat().st_size / 1024**2, 2), 'path': str(path)
    })
print(json.dumps(audit, indent=2, ensure_ascii=False))
